# Extracción de Datos

En este notebook se busca extraer la información necesaria para ser consumida por los modelos.

Variables a considerar:
- Aportes hídricos
- Compras en bolsa
- Compras en contratos
- Demanda del SIN
- Disponibilidad Declarada
- Disponibilidad Real
- Generación real
- Indice ONI
- Precio de bolsa
- Precio de oferta
- Precio de Escasez
- Reservas hídricas
- Ventas en bolsa
- Ventas en contratos
- Vertimientos


In [2]:
# ============================================================
# PASO 0: Instalación de dependencias (solo la primera vez / entorno nuevo)
# ============================================================
# Se deja comentado porque el entorno local ya tiene pydataxm instalado.
# Si se corre en un entorno nuevo (ej. Google Colab o una máquina limpia),
# descomentar la siguiente línea para instalar la librería oficial de XM.
# !pip install pydataxm
# !pip list   # lista todos los paquetes instalados, útil para depurar versiones

In [3]:
# ============================================================
# PASO 1: Importación de librerías y configuración del entorno de trabajo
# ============================================================
import datetime as dt                    # manejo de fechas (start_date, end_date, etc.)
import pydataxm                          # librería oficial de XM para consultar su API pública
from pydataxm.pydataxm import ReadDB     # clase principal de consulta (get_collections, request_data)
import pandas as pd                      # manipulación de datos tabulares (DataFrames)
import pathlib                           # manejo de rutas de archivos multiplataforma
import os                                # operaciones de sistema de archivos (crear carpetas, unir rutas)
import time                              # pausas entre solicitudes a la API (time.sleep)
# os.chdir(pathlib.Path.cwd())

# --- Bloque específico de Google Colab (desactivado para ejecución local) ---
# Si se corre en Colab, se monta Google Drive para persistir los datos descargados
# y se cambia el directorio de trabajo a la carpeta del proyecto dentro de Drive.
# Conexión de google drive con el repo
#from google.colab import drive
#drive.mount('/content/drive')

###==========================
#ruta = os.path.join("/content/drive/MyDrive", "MonografiaUdeA")
#os.chdir(ruta)
# print("Ruta:", ruta)

# Directorio de trabajo actual (carpeta donde vive este notebook).
# Todas las rutas del notebook (datasets/, metrics.csv, etc.) se construyen a partir de esta.
abspath = pathlib.Path.cwd()
print(f"Directorio de trabajo: {abspath}")

# Función propia del repo (_periods.py) que trocea un rango de fechas largo en
# bloques más pequeños (ej. de 12 meses). Es necesaria porque la API de XM limita
# cuántos días se pueden pedir por consulta (ver columna MaxDays en metrics.csv).
from _periods import get_periods
###==========================

Directorio de trabajo: c:\Users\Windows 10 Pro\OneDrive\Trabajo Integrador\MonografiaUdeA-main


In [4]:
# ============================================================
# PASO 2: Descargar el catálogo de métricas disponibles en la API de XM
# ============================================================
# ReadDB() crea el objeto de conexión a la API de XM (es pública, no requiere credenciales).
consult = ReadDB()

# get_collections() devuelve el catálogo completo de métricas que ofrece la API
# (MetricId, MetricName, Entity, MaxDays, Type, Url, Filter, MetricUnits, MetricDescription).
# Se ordena alfabéticamente por nombre y se guarda en metrics.csv.
#
# ⚠️ IMPORTANTE: este metrics.csv es el CATÁLOGO de la API de XM, y se sobrescribe
# cada vez que se corre esta celda. NO es el registro de resultados/métricas de los
# modelos (MAE, RMSE, SMAPE) que se documenta por separado — no confundir ambos usos
# del mismo nombre de archivo (ver hallazgo H1 del análisis de estructura del repo).
consult.get_collections().sort_values(by = 'MetricName').to_csv(os.path.join(abspath, "metrics.csv"), index=False)

## Parámetros para la extracción de la información

In [ ]:
# ============================================================
# PASO 3: Parámetros de la extracción — rango de fechas y variables a descargar
# ============================================================
# AJUSTE 2026-09-21: extracción automática. Para actualizar los datos ya no hay que editar
# fechas ni comentar o descomentar variables: basta con volver a correr el notebook.
#
# La fecha final es AYER: XM publica con uno o dos días de rezago, y si el último día todavía no
# está disponible la API simplemente no lo devuelve. El periodo de ESTUDIO no se fija aquí sino en
# 2_DataProcessing.ipynb (FECHA_FIN_ESTUDIO), para que actualizar los datos crudos no cambie por
# accidente el periodo que analiza el documento.
start_date = dt.date(2010, 1, 1)
end_date = dt.date.today() - dt.timedelta(days=1)

# "incremental": en cada CSV solo se descarga lo que falta, repitiendo los últimos DIAS_REVISION
#                días porque XM ajusta las cifras recientes (ver el Paso 5).
# "completo":    se descarga todo desde start_date, como se hacía antes.
MODO = "incremental"
DIAS_REVISION = 90

# La API de XM limita el rango de días por solicitud (ver columna MaxDays en metrics.csv), por eso
# cada descarga se trocea en periodos de 12 meses con get_periods().
months_per_period = 12
periods = get_periods(start_date, end_date, months_per_period)
print(f"Extracción hasta {end_date}, en modo {MODO}")

# Carpeta donde se guardan todos los CSV crudos descargados.
dataset_folder_name = "datasets"

# Catálogo de variables a extraer, como pares [MetricId, Entity]:
# - MetricId: identificador de la métrica en la API de XM (coincide con metrics.csv)
# - Entity:   nivel de agregación ("Sistema" = agregado nacional,
#             "Recurso" = desagregado por planta de generación)
# Quedan activas las 13 que usa 2_DataProcessing.ipynb. DemaSIN y DispoReal se retiraron de las
# variables candidatas (ver el documento) y quedan comentadas.
metricas = [
     ["AporEner", "Sistema"],         # Aportes hídricos de energía al sistema
     ["CompBolsNaciEner", "Sistema"], # Compras de energía en bolsa nacional
     ["CompContEner", "Sistema"],     # Compras de energía por contratos
     ["DemaReal", "Sistema"],         # Demanda real de energía
     #["DemaSIN", "Sistema"],         # Demanda del SIN (retirada de las candidatas)
     ["DispoDeclarada", "Recurso"],   # Disponibilidad declarada, por recurso/planta
     #["DispoReal", "Recurso"],       # Disponibilidad real (retirada de las candidatas)
     ["Gene", "Recurso"],             # Generación real, por recurso/planta
     ["PrecBolsNaci", "Sistema"],     # Precio de bolsa nacional (variable objetivo)
     ["PrecOferDesp", "Recurso"],     # Precio de oferta despachado, por recurso/planta
     ["VentBolsNaciEner", "Sistema"], # Ventas de energía en bolsa nacional
     ["VentContEner", "Sistema"],     # Ventas de energía por contratos
     ["VoluUtilDiarEner", "Sistema"], # Volumen útil diario de los embalses
     ["VertEner", "Sistema"],         # Vertimientos de energía del sistema
     ["PrecEsca", "Sistema"],         # Precio de Escasez (CREG) — proxy de combustibles.
                                      # Se prefirió sobre desagregar PrecOferDesp por tipo
                                      # de combustible (esa vía se descartó: redundante con
                                      # precio_oferta ya existente, corr. 0.765 con
                                      # precio_bolsa). PrecEsca tiene cobertura completa
                                      # 2010-2026 vía esta misma API, y baja correlación con
                                      # precio_oferta (0.136) y precio_bolsa (-0.211), por lo
                                      # que no es redundante con lo ya incluido. El iGas-D
                                      # (precio de mercado del gas, fuente externa BEC/BMC,
                                      # cobertura 2019+) sigue evaluándose aparte, sin
                                      # descartarse — ver bitácora, sección 9.
]

## Obtener listado de los recursos de generación

In [6]:
# ============================================================
# PASO 4: Obtener el listado de recursos de generación (catálogo de plantas)
# ============================================================
# Este listado es necesario más adelante (en 2_DataProcessing.ipynb) para poder
# clasificar cada planta como hidráulica, térmica, etc., y así poder agregar
# variables como Gene o DispoDeclarada por tipo de recurso.

# ===============================
# Configuración de rutas
# ===============================

dataset_folder_name = "datasets"

ruta_datasets = os.path.join(abspath, dataset_folder_name)

# Crea la carpeta datasets/ si todavía no existe.
if not os.path.exists(ruta_datasets):
    os.makedirs(ruta_datasets)

# ===============================
# Función para obtener recursos
# ===============================

def obtener_recursos():
    # Nueva conexión a la API (una por función, para evitar reutilizar un objeto
    # que pueda quedar en mal estado tras muchas solicitudes).
    consult = ReadDB()

    # ListadoRecursos: catálogo de todas las plantas de generación del SIN,
    # con su tipo de tecnología (hidráulica, térmica, etc.).
    recursos_generacion = consult.request_data(
        "ListadoRecursos",
        "Sistema",
        start_date,
        end_date
    )

    ruta_archivo = os.path.join(ruta_datasets, "listado_recursos.csv")

    # Sobrescribe el archivo completo cada vez que se corre (no es incremental).
    recursos_generacion.to_csv(ruta_archivo, index=False)

    print("Archivo actualizado en:")
    print(ruta_archivo)
    print("Filas descargadas:", len(recursos_generacion))

    return recursos_generacion


# ===============================
# EJECUTAR FUNCIÓN
# ===============================

recursos_generacion = obtener_recursos()

Archivo actualizado en:
c:\Users\Windows 10 Pro\OneDrive\Trabajo Integrador\MonografiaUdeA-main\datasets\listado_recursos.csv
Filas descargadas: 2298


## Obtener las demás variables

In [ ]:
# ============================================================
# PASO 5: Extraer las variables de XM listadas en "metricas" (Paso 3)
# ============================================================
# AJUSTE 2026-09-21: extracción automática e incremental.
# - Si el CSV de la variable no existe, o si MODO == "completo", se descarga todo desde
#   start_date hasta end_date.
# - Si ya existe, solo se descarga desde DIAS_REVISION días antes de su última fecha hasta
#   end_date, y ese tramo reemplaza al guardado: XM ajusta las cifras de los meses recientes, así
#   que conviene volver a traerlos. Todo lo anterior se conserva tal cual.
# Por cada variable se descarga bloque por bloque (12 meses) y se guarda un CSV en datasets/.

if not os.path.exists(os.path.join(abspath, dataset_folder_name)):
    os.makedirs(os.path.join(abspath, dataset_folder_name))


def rango_pendiente(ruta_csv):
    """Devuelve (fecha desde la que hay que descargar, filas previas que se conservan)."""
    if MODO == "completo" or not os.path.exists(ruta_csv):
        return start_date, None
    previo = pd.read_csv(ruta_csv)
    fechas = pd.to_datetime(previo["Date"])
    desde = max(start_date, fechas.max().date() - dt.timedelta(days=DIAS_REVISION))
    return desde, previo[fechas.dt.date < desde]


def extraer():
    for metrica, entidad in metricas:
        ruta = os.path.join(abspath, dataset_folder_name, f"{metrica}.csv")
        desde, previo = rango_pendiente(ruta)
        if desde > end_date:
            print(f"{metrica}: al día")
            continue
        print(f"{metrica}: descargando de {desde} a {end_date}")
        consult = ReadDB()                       # nueva conexión por variable
        partes = []
        for inicio, fin in get_periods(desde, end_date, months_per_period):
            partes.append(consult.request_data(metrica, entidad, inicio, fin))
            time.sleep(2)                        # pausa entre solicitudes (no saturar la API)
        nuevo = pd.concat(partes)
        if nuevo.empty:
            print("   la API no devolvió filas nuevas")
            continue
        # La fecha se guarda con el mismo formato de texto que ya tienen los CSV (AAAA-MM-DD).
        nuevo["Date"] = pd.to_datetime(nuevo["Date"]).dt.strftime("%Y-%m-%d")
        total = nuevo if previo is None else pd.concat([previo, nuevo])
        total.to_csv(ruta, index=False)
        print(f"   {len(nuevo)} filas descargadas; {len(total)} en total")


extraer()

## Obtener ONI

In [ ]:
# ============================================================
# PASO 6: Obtener el índice ONI (Oceanic Niño Index) desde NOAA — CORREGIDO 2026-08-09
# ============================================================
# El índice ONI NO viene de la API de XM sino directamente de NOAA, ya que mide
# la anomalía de temperatura del Pacífico asociada al fenómeno El Niño/La Niña,
# que impacta la hidrología (aportes hídricos) y por lo tanto el precio de bolsa.
#
# CAMBIOS respecto a la versión anterior:
# 1) La URL original (origin.cpc.ncep.noaa.gov/.../ensostuff/oni.ascii.txt) ya no
#    resuelve por DNS (NOAA dio de baja/movió ese dominio). Se reemplaza por el
#    endpoint vigente, el mismo que respalda la tabla oficial. Hoy NOAA
#    calcula el ONI con ERSST.v6 (la tabla v5 queda como histórico):
#    https://www.cpc.ncep.noaa.gov/products/analysis_monitoring/enso/oni/v6/
# 2) El formato real de NOAA nunca tuvo una columna "MON" (mes individual) como
#    asumía la versión anterior. Trae columnas SEAS, YR, TOTAL, ANOM, donde SEAS
#    es una temporada móvil de 3 meses (DJF, JFM, FMA, ..., NDJ) — el ONI está
#    definido así por naturaleza. Se mapea cada temporada a su mes central,
#    replicando en código la misma convención que se usó al mantener a mano
#    datasets/Indice_ONI.xlsx.
# 3) La salida ahora se genera en formato ANCHO (Year + columnas de mes 1-12),
#    igual a datasets/Indice_ONI.xlsx, y se guarda con ese mismo nombre — así
#    2_DataProcessing.ipynb sigue funcionando sin ningún cambio, y el índice ONI
#    queda a utomatizado en vez de depender de la actualización manual.
# 4) AJUSTE 2026-09-21: se conserva desde 2009 (ver el Paso 1 de esta celda).
import pandas as pd
import os
import datetime as dt

# ===============================
# 1) Configuración de fechas
# ===============================
# Se conserva desde 2009, un año antes que el resto del dataset (AJUSTE 2026-09-21).
# El periodo de estudio empieza en 2010, pero la versión «publicada» del ONI que
# construye 2_DataProcessing.ipynb necesita las temporadas de finales de 2009: en
# enero y febrero de 2010, el último valor que NOAA había publicado era el de esas
# temporadas. 2_DataProcessing.ipynb recorta después al periodo de estudio.
anio_inicio = 2009

# ===============================
# 2) Descargar ONI oficial NOAA (endpoint vigente)
# ===============================
url = "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"

oni_raw = pd.read_csv(url, sep=r"\s+")
print("ONI descargado correctamente desde NOAA")
print(oni_raw.tail())

# ===============================
# 3) Mapear cada temporada de 3 meses a su mes central
# ===============================
# Convención oficial de la tabla de NOAA: cada temporada (SEAS) se asigna a la
# columna de su mes central (DJF -> enero, JFM -> febrero, ..., NDJ -> diciembre).
SEASON_TO_MONTH = {
    "DJF": 1, "JFM": 2, "FMA": 3, "MAM": 4,
    "AMJ": 5, "MJJ": 6, "JJA": 7, "JAS": 8,
    "ASO": 9, "SON": 10, "OND": 11, "NDJ": 12,
}

oni = oni_raw.rename(columns={"YR": "anio", "ANOM": "oni"})
oni["mes"] = oni["SEAS"].map(SEASON_TO_MONTH)
oni = oni[oni["anio"] >= anio_inicio]

# ===============================
# 4) Pivotear a formato ancho (Year x mes 1-12)
# ===============================
# Mismo formato que hoy tiene datasets/Indice_ONI.xlsx, para que
# 2_DataProcessing.ipynb lo siga leyendo sin ningún cambio.
indice_ONI = oni.pivot(index="anio", columns="mes", values="oni").reset_index()
indice_ONI = indice_ONI.rename(columns={"anio": "Year"})
indice_ONI = indice_ONI[["Year"] + list(range(1, 13))]  # orden explícito de columnas 1..12

print("\nONI (formato ancho Year x mes) generado correctamente")
print(indice_ONI.tail())

# ===============================
# 5) Guardar
# ===============================
# ⚠️ Esto SOBRESCRIBE datasets/Indice_ONI.xlsx cada vez que se corre esta celda,
# reemplazando la actualización manual que se venía haciendo. Si se prefiere no
# sobrescribir automáticamente, cambiar el nombre de archivo antes de correr.
ruta_guardado = os.path.join(abspath, "datasets", "Indice_ONI.xlsx")
indice_ONI.to_excel(ruta_guardado, index=False)

print("\nArchivo ONI guardado en:")
print(ruta_guardado)

## Obtener IPC e IPP (DANE)

In [ ]:
# ============================================================
# PASO 7: IPC e IPP desde el DANE — descarga automática (AJUSTE 2026-09-21)
# ============================================================
# Antes estos dos archivos se bajaban a mano de las páginas web; ahora se descargan aquí.
# El DANE publica cada mes un anexo con la serie completa, y el nombre del archivo lleva el mes
# de referencia (por ejemplo, anex-IPP-historicos-ago2026.xlsx). Como no se sabe de antemano cuál
# es el último publicado, se prueba desde el mes actual hacia atrás hasta encontrarlo.
#
# - IPC: anexo «Índices - serie de empalme» (base diciembre 2018 = 100). Es la misma serie que
#   difunde el Banco de la República, que la toma del DANE: se verificó que coinciden en todos
#   los meses desde 2009.
# - IPP: anexo «IPP Histórico» (base diciembre 2014 = 100), el mismo formato que ya leía
#   2_DataProcessing.ipynb.
import requests

MESES_ABR = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
ANEXOS_DANE = {
    "IPC_DANE.xlsx": "https://www.dane.gov.co/files/operaciones/IPC/{mes}/anex-IPC-Indices-{mes}.xlsx",
    "IPP_DANE.xlsx": "https://www.dane.gov.co/files/operaciones/IPP/anex-IPP-historicos-{mes}.xlsx",
}


def ultimo_anexo_dane(plantilla, meses_atras=12):
    """Descarga el anexo más reciente publicado. Un .xlsx válido empieza por 'PK' (es un zip)."""
    hoy = pd.Timestamp(dt.date.today())
    for k in range(meses_atras + 1):
        f = hoy - pd.DateOffset(months=k)
        mes = f"{MESES_ABR[f.month - 1]}{f.year}"
        url = plantilla.format(mes=mes)
        r = requests.get(url, timeout=120)
        if r.status_code == 200 and r.content[:2] == b"PK":
            return url, r.content
    raise RuntimeError(f"No hay anexo publicado en los últimos {meses_atras} meses: {plantilla}")


for nombre, plantilla in ANEXOS_DANE.items():
    url, contenido = ultimo_anexo_dane(plantilla)
    with open(os.path.join(abspath, "datasets", nombre), "wb") as fh:
        fh.write(contenido)
    print(f"{nombre} <- {url}")